# Notebook 5 : Web scraping

In [13]:
# Décommenter la ligne suivante pour installer lxml (nécessaire pour read_html)
#%pip install lxml

In [14]:
import pandas as pd
import requests

from bs4 import BeautifulSoup

## Bonsaïs

Le site [Umi Zen Bonsai](https://umizenbonsai.com/) est une boutique de vente en ligne dédiée aux bonsaïs. Les conifères sont disponible sur la page web [https://umizenbonsai.com/shop/bonsai/coniferes/](https://umizenbonsai.com/shop/bonsai/coniferes/). Comme beaucoup d'autres sites, l'information est organisée en blocs dans lesquels il est possible de récupérer des données.

Pour scraper ce type de site, le processus consiste à capturer les blocs dans un premier temps, puis à en extraire les données.

1. Récupérer le contenu de la page avec `requests` et passer le résultat au parser de `BeautifulSoup`.

In [15]:
base_url = "https://umizenbonsai.com/shop/bonsai/coniferes/"
page_url=""

url=base_url+page_url

r=requests.get(url)
r.raise_for_status()

print(r.text[:800] + "...")

<!DOCTYPE html>
<html class="html" lang="fr-FR">
<head>
	<meta charset="UTF-8">
	<link rel="profile" href="https://gmpg.org/xfn/11">

	<meta name='robots' content='index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1' />
<link rel="preload" as="font" href="https://fonts.gstatic.com/s/montserrat/v26/JTUQjIg1_i6t8kCHKm459WxRyS7m0dR9pA.woff2" data-wpacu-preload-google-font="1" crossorigin>
<link rel="preload" as="font" href="https://fonts.gstatic.com/s/montserrat/v26/JTUSjIg1_i6t8kCHKm459WlhyyTh89Y.woff2" data-wpacu-preload-google-font="1" crossorigin>
<link rel="preload" as="font" href="https://fonts.gstatic.com/s/montserrat/v26/JTUSjIg1_i6t8kCHKm459Wlhyw.woff2" data-wpacu-preload-google-font="1" crossorigin>
<meta name="viewport" content="width=device-width, initial-...


In [16]:
soup = BeautifulSoup(r.text, "html.parser") # HTML parser de Python
print(str(soup)[:700] + "...")


<!DOCTYPE html>

<html class="html" lang="fr-FR">
<head>
<meta charset="utf-8"/>
<link href="https://gmpg.org/xfn/11" rel="profile"/>
<meta content="index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1" name="robots">
<link as="font" crossorigin="" data-wpacu-preload-google-font="1" href="https://fonts.gstatic.com/s/montserrat/v26/JTUQjIg1_i6t8kCHKm459WxRyS7m0dR9pA.woff2" rel="preload"/>
<link as="font" crossorigin="" data-wpacu-preload-google-font="1" href="https://fonts.gstatic.com/s/montserrat/v26/JTUSjIg1_i6t8kCHKm459WlhyyTh89Y.woff2" rel="preload"/>
<link as="font" crossorigin="" data-wpacu-preload-google-font="1" href="https://fonts.gstatic.com/s/montserrat/v26/...


2. Écrire un sélecteur CSS pour capturer les éléments `li` qui contiennent les blocs correspondants aux bonsaïs. Vérifier sur le site que le nombre de bonsaïs affichés correspond.

In [19]:
css_selector = "ul.products li.entry"
bonsai= soup.select(css_selector)

print('nb objet '+str(len(bonsai)))

nb objet 5


In [21]:
print(bonsai[0])

<li class="entry has-media col span_1_of_4 owp-content-center owp-thumbs-layout-horizontal owp-btn-big owp-tabs-layout-horizontal circle-sale product type-product post-55126 status-publish first instock product_cat-coniferes product_cat-tous-les-bonsai product_tag-bonsai has-post-thumbnail taxable shipping-taxable product-type-simple">
<div class="product-inner clr">
<div class="woo-entry-image clr">
<a class="woocommerce-LoopProduct-link" href="https://umizenbonsai.com/vente/bonsai-cypres-du-japon/"><img alt="Bonsai Cyprès du Japon – 33cm" class="woo-entry-image-main" decoding="async" fetchpriority="high" height="800" itemprop="image" sizes="(max-width: 800px) 100vw, 800px" src="https://umizenbonsai.com/wp-content/uploads/bonsai-cypres-du-japon-umi-zen-bonsai-shop.jpg" srcset="https://umizenbonsai.com/wp-content/uploads/bonsai-cypres-du-japon-umi-zen-bonsai-shop.jpg 800w, https://umizenbonsai.com/wp-content/uploads/bonsai-cypres-du-japon-umi-zen-bonsai-shop-150x150.jpg 150w, https://u

In [22]:
print(bonsai[0].select('li.price-wrap bdi'))

[<bdi>150.00<span class="woocommerce-Price-currencySymbol">€</span></bdi>]


In [25]:
print(bonsai[0].select('li.price-wrap bdi')[0].text)

150.00€


In [27]:
print(bonsai[0].select('li.price-wrap bdi')[0].text[:-1])

150.00


3. Écrire une fonction qui prend un bloc de la liste précédente et retourne un tuple contenant le nom, le prix et le lien de description du bonsaï.

In [ ]:
#tentative recuperation du nom
print(bonsai[0].select('li.title a'))

[<a href="https://umizenbonsai.com/vente/bonsai-cypres-du-japon/">Bonsai Cyprès du Japon – 33cm</a>]


In [30]:
print(bonsai[0].select('li.title a')[0].text)

Bonsai Cyprès du Japon – 33cm


In [31]:
#recuperation lien
print(bonsai[0].select('li.title a')[0].attrs('href'))

TypeError: 'AttributeDict' object is not callable

In [35]:
def bonsai_data(bonsai):
    bonsai_a=bonsai.select('li.title a')[0]
    nom=bonsai_a.text
    url=bonsai_a.attrs['href']
    prix_str=bonsai.select('li.price-wrap bdi')[0].text
    return {'nom':nom,'url':url,'prix':prix_str}

In [39]:
bonsai_df=pd.DataFrame([bonsai_data(b) for b in bonsai])

bonsai_df.head(10)

,nom,url,prix
0,Bonsai Cyprès du Japon – 33cm,https://umizenbonsai.com/vente/bonsai-cypres-d...,150.00€
1,Bonsai Genévrier de Phénicie – 67cm,https://umizenbonsai.com/vente/bonsai-genevrie...,450.00€
2,Bonsai If Commun – 38cm,https://umizenbonsai.com/vente/bonsai-if-commu...,450.00€
3,Bonsai If Commun – 58cm,https://umizenbonsai.com/vente/bonsai-if-commu...,"1,350.00€"
4,Bonsai Pin Sylvestre – 38cm,https://umizenbonsai.com/vente/bonsai-pin-sylv...,"1,250.00€"


4. Utiliser les deux questions précédentes pour construire un dataframe contenant les données des bonsaïs.

5. (*Bonus*) Écrire une fonction pour récupérer la provenance, le feuilage et les dimension du bonsaï à partir du lien de description. Utiliser cette fonction pour ajouter des colonnes au dataframe précédent

## Trampoline

Le trampoline est un sport olympique depuis les jeux de Sydney en 2000. La page suivante contient les listes des hommes et des femmes ayant obtenu une médaille olympique dans cette discipline :
[https://fr.wikipedia.org/wiki/Liste_des_m%C3%A9daill%C3%A9s_olympiques_au_trampoline](https://fr.wikipedia.org/wiki/Liste_des_m%C3%A9daill%C3%A9s_olympiques_au_trampoline)

Un tableau est contenu dans un élément `table` avec des balises pour les lignes `tr`, pour les colonnes `th`, pour les cellules `td`, ... Cela peut être fastidieux à scraper et très répétitif. Heureusement, Pandas propose la fonction `read_html` pour récupérer des tableaux sous forme de dataframes à partir d'une page web.

1. Récupérer le contenu de la page des médaillés olympiques au trampoline avec `requests` en passant un *User Agent* dans les en-têtes `headers`. Utiliser ensuite la fonction `read_html` de Pandas directement sur le contenu textuel de la page. Combien de dataframes sont récupérés ?

2. Extraire de la liste précédente les dataframes des médailles masculines et féminines.

3. À partir de ces dataframes, compter combien chaque pays a reçu de médailles d'or, d'argent et de bronze.

4. (*Bonus*) Construire un dataframe contenant, pour chaque pays, le nombre de médailles d'or, d'argent et de bronze ainsi que le nombre total de médailles. Classer ce dataframe dans l'ordre usuel en fonction d'abord du nombre de médailles d'or, puis du nombre de médailles d'argent et enfin du nombre de médailles de bronze. Comparer le résultat avec le tableau des médailles sur la page [https://fr.wikipedia.org/wiki/Trampoline_aux_Jeux_olympiques](https://fr.wikipedia.org/wiki/Trampoline_aux_Jeux_olympiques).

## Cate Blanchett

Dans le cours, nous avons essayé de trouver avec quels acteurs Cate Blanchett a joué le plus au cours des années 2000. Pour cela, nous avons récupéré la liste des pages Wikipedia des films où elle tient un rôle avec le code suivant :

In [32]:
url_wikipedia = "https://fr.wikipedia.org"
url_blanchett = url_wikipedia + "/wiki/Cate_Blanchett"

# Ne pas oublier le User Agent pour les pages Wikipedia
r_blanchett = requests.get(url_blanchett, headers={"User-Agent": "CepeBot"})
assert r_blanchett.status_code == 200, f"Erreur {r_blanchett.status_code}"

soup_blanchett = BeautifulSoup(r_blanchett.text, "html.parser")

selector_films = "#mw-content-text div ul:nth-of-type(3) li i a"
films_blanchett = soup_blanchett.select(selector_films)

films_data = [
    {
        "titre": film.attrs["title"],
        "url_wikipedia": url_wikipedia + film.attrs["href"]
    }
    for film in films_blanchett
    if not (
        film.attrs.get("class") == ["new"] # Film sans page
        or film.attrs["title"] == "Galadriel" # Mauvais lien
    )
]

films = pd.DataFrame(films_data)

films

,titre,url_wikipedia
0,Les Larmes d'un homme,https://fr.wikipedia.org/wiki/Les_Larmes_d%27u...
1,Intuitions,https://fr.wikipedia.org/wiki/Intuitions
2,"Bandits (film, 2001)","https://fr.wikipedia.org/wiki/Bandits_(film,_2..."
3,Le Seigneur des anneaux : La Communauté de l'a...,https://fr.wikipedia.org/wiki/Le_Seigneur_des_...
4,Charlotte Gray,https://fr.wikipedia.org/wiki/Charlotte_Gray
5,Terre Neuve (film),https://fr.wikipedia.org/wiki/Terre_Neuve_(film)
6,"Heaven (film, 2002)","https://fr.wikipedia.org/wiki/Heaven_(film,_2002)"
7,Le Seigneur des anneaux : Les Deux Tours,https://fr.wikipedia.org/wiki/Le_Seigneur_des_...
8,Veronica Guerin (film),https://fr.wikipedia.org/wiki/Veronica_Guerin_...
9,Coffee and Cigarettes,https://fr.wikipedia.org/wiki/Coffee_and_Cigar...


Le sélecteur CSS que nous avons utilisé ne permettait pas d'obtenir la réponse à notre question car il ne capturait pas toutes les listes d'acteurs (organisation différente pour *Coffee and Cigarettes*, double colonne pour *Aviator*, ...). En effet, les pages Wikipedia des films ne sont pas uniformes et il n'est pas possible d'extraire la distribution de tous les films avec le même sélecteur.

Pour remédier à cela, nous proposons ici d'aller scraper la liste des acteurs sur le site [TMDB](https://www.themoviedb.org/) (*The Movie Database*) dont les pages obéissent toutes à la même organisation. Les pages Wikipedia relatives à des films contiennent toutes un lien externe vers ce site.

1. Pour chaque film, scraper la page Wikipedia pour récupérer le lien vers la page TMDB associée et déduire le lien du casting complet qui ser ajouté le dans une nouvelle colonne du dataframe `films`.

In [34]:
url=films.iat[3, 1]
print(url)

https://fr.wikipedia.org/wiki/Le_Seigneur_des_anneaux_:_La_Communaut%C3%A9_de_l%27anneau


In [40]:
r=requests.get(url, headers={"User-Agent": "CepeBot_EC"})
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser") # HTML parser de Python
print(str(soup)[:300] + "...")

<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-menu-disabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disa...


In [43]:
#css_selector
link_list_tmdb= soup.select('a.external')

for elem in link_list_tmdb:
    print(elem)


<a class="external text" href="https://fr.wikipedia.org/w/index.php?title=Le_Seigneur_des_anneaux_:_La_Communaut%C3%A9_de_l%27anneau&amp;action=edit">modifier</a>
<a class="external text" href="https://fr.wikipedia.org/w/index.php?title=Le_Seigneur_des_anneaux_:_La_Communaut%C3%A9_de_l%27anneau&amp;action=edit">modifier</a>
<a class="external text" href="http://www.filmdeculte.com/cinema/film/Seigneur-des-anneaux-La-Communaute-de-lanneau-Le-1373.html" rel="nofollow"><cite style="font-style:normal;">Seigneur des anneaux : La Communauté de l'anneau (Le)</cite></a>
<a class="external text" href="https://voiretmanger.fr/seigneur-anneaux-communaute-anneau-jackson/" rel="nofollow"><cite style="font-style:normal;">Le Seigneur des anneaux : la Communauté de l’anneau, Peter Jackson</cite></a>
<a class="external text" href="https://www.avcesar.com/test/bluray/id-602/le-seigneur-des-anneaux-la-communaute-de-lanneau.html" rel="nofollow"><cite style="font-style:normal;">Le seigneur des anneaux : la

In [44]:
found_url=None
for external_link in link_list_tmdb:
    external_url=external_link.attrs['href']
    if 'themoviedb' in external_url:
        found_url=external_url
        break

print(found_url)

https://www.themoviedb.org/movie/120


In [45]:
#on definit la fonction

def get_tmdb_url(film_url):
    r=requests.get(url, headers={"User-Agent": "CepeBot_EC"})
    r.raise_for_status()
    film_soup = BeautifulSoup(r.text, "html.parser")
    found_url=None
    for external_link in link_list_tmdb:
        external_url=external_link.attrs['href']
        if 'themoviedb' in external_url:
            found_url=external_url
            break
    return found_url

2. La liste des acteurs d'un film se présente comme une liste ordonnée `ol` dans les pages TMDB. Scraper les pages de casting pour ajouter la liste des acteurs de chaque film dans une nouvelle colonne du dataframe `films`.

In [48]:
films['tmdb_url']=films['url_wikipedia'].apply(get_tmdb_url)
films['tmdb_cast_url']=(films['tmdb_url'].apply(lambda tmdb_url:tmdb_url+"/cast" if tmdb_url is not None else None))

films

,titre,url_wikipedia,tmdb_url,tmdb_cast_url
0,Les Larmes d'un homme,https://fr.wikipedia.org/wiki/Les_Larmes_d%27u...,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
1,Intuitions,https://fr.wikipedia.org/wiki/Intuitions,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
2,"Bandits (film, 2001)","https://fr.wikipedia.org/wiki/Bandits_(film,_2...",https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
3,Le Seigneur des anneaux : La Communauté de l'a...,https://fr.wikipedia.org/wiki/Le_Seigneur_des_...,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
4,Charlotte Gray,https://fr.wikipedia.org/wiki/Charlotte_Gray,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
5,Terre Neuve (film),https://fr.wikipedia.org/wiki/Terre_Neuve_(film),https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
6,"Heaven (film, 2002)","https://fr.wikipedia.org/wiki/Heaven_(film,_2002)",https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
7,Le Seigneur des anneaux : Les Deux Tours,https://fr.wikipedia.org/wiki/Le_Seigneur_des_...,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
8,Veronica Guerin (film),https://fr.wikipedia.org/wiki/Veronica_Guerin_...,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast
9,Coffee and Cigarettes,https://fr.wikipedia.org/wiki/Coffee_and_Cigar...,https://www.themoviedb.org/movie/120,https://www.themoviedb.org/movie/120/cast


In [51]:
r=requests.get(films.iat[3,3], headers={"User-Agent": "CepeBot_EC"})
r.raise_for_status()
test_soup = BeautifulSoup(r.text, "html.parser")
print(str(test_soup)[:300] + "...")

<!DOCTYPE html>

<html class="no-js" lang="fr">
<head>
<title>Le Seigneur des anneaux : La Communauté de l'anneau (2001) - Distribution des rôles et équipe technique — The Movie Database (TMDB)</title>
<meta content="on" http-equiv="cleartype"/>
<meta charset="utf-8"/>
<meta content="Movies, TV Show...


In [54]:
people_list=test_soup.select('ol.people.credits')
for elem in people_list:
    print(elem)

<ol class="people credits">
<li data-credit-id="52fe421ac3a36847f800448f" data-order="0">
<a href="/person/109-elijah-wood">
<div class="glyphicons_v2 picture grey profile no_image_holder two">
<img alt="Elijah Wood" class="profile w-full" loading="lazy" src="https://media.themoviedb.org/t/p/w66_and_h66_face/ayARmqAe9Aab1zg6FjJG0u9MEBo.jpg" srcset="https://media.themoviedb.org/t/p/w66_and_h66_face/ayARmqAe9Aab1zg6FjJG0u9MEBo.jpg 1x, https://media.themoviedb.org/t/p/w132_and_h132_face/ayARmqAe9Aab1zg6FjJG0u9MEBo.jpg 2x"/>
</div>
</a>
<div>
<div class="info">
<p><a href="/person/109-elijah-wood">Elijah Wood</a></p>
<p class="character">
              Frodo
            </p>
</div>
</div>
</li>
<li data-credit-id="52fe421ac3a36847f8004493" data-order="1">
<a href="/person/1327-ian-mckellen">
<div class="glyphicons_v2 picture grey profile no_image_holder two">
<img alt="Ian McKellen" class="profile w-full" loading="lazy" src="https://media.themoviedb.org/t/p/w66_and_h66_face/coWjgMEYJjk2OrN

In [56]:
actor_list=[p for p in people_list if "crew" not in p.attrs['class']]
print(actor_list)

[<ol class="people credits">
<li data-credit-id="52fe421ac3a36847f800448f" data-order="0">
<a href="/person/109-elijah-wood">
<div class="glyphicons_v2 picture grey profile no_image_holder two">
<img alt="Elijah Wood" class="profile w-full" loading="lazy" src="https://media.themoviedb.org/t/p/w66_and_h66_face/ayARmqAe9Aab1zg6FjJG0u9MEBo.jpg" srcset="https://media.themoviedb.org/t/p/w66_and_h66_face/ayARmqAe9Aab1zg6FjJG0u9MEBo.jpg 1x, https://media.themoviedb.org/t/p/w132_and_h132_face/ayARmqAe9Aab1zg6FjJG0u9MEBo.jpg 2x"/>
</div>
</a>
<div>
<div class="info">
<p><a href="/person/109-elijah-wood">Elijah Wood</a></p>
<p class="character">
              Frodo
            </p>
</div>
</div>
</li>
<li data-credit-id="52fe421ac3a36847f8004493" data-order="1">
<a href="/person/1327-ian-mckellen">
<div class="glyphicons_v2 picture grey profile no_image_holder two">
<img alt="Ian McKellen" class="profile w-full" loading="lazy" src="https://media.themoviedb.org/t/p/w66_and_h66_face/coWjgMEYJjk2Or

In [ ]:
actor=[]
for a in actor_list:
    for actor_item in actor_list.selec('div.info a'):
        actor.append(actor_item)

3. Utiliser le résultat de la question précédente pour répondre à la question initiale : avec quels acteurs Cate Blanchett a-t-elle partagé l'affiche le plus souvent au cours des années 2000 ?